In [55]:
import numpy as np
from transformers import pipeline

# Thougts on improvement and expansion of scope
""" Implement Conditional statements to ensure all parties give arguments that follow the certain predefined conditions.(eg: statements must not contradict the geneva conventions) """
""" Implement grounding nodes as evidence for certain claims """
""" Implement verifiability for uncertain claims: Upload report or pdf that can be used for verifying claims (also can use claimify) """
""" Diplomatic use case: Ensure statement is consistent with old statements(done) and does not contradicts ally's stance """
""" Acceptability of a node (Dung Smantics). A node N is said to be acceptable wrt set S if all attackers of N have attackers in set S -> 
This can be used to identify the claims that are unsupported and to be focused on during effective argumentation"""


def find_cycles_adj_matrix(adj_matrix):
    n = len(adj_matrix)
    visited = [False] * n
    stack = [False] * n
    cycles = []
    
    def dfs(v, path):
        visited[v] = True
        stack[v] = True
        path.append(v)

        for u in range(n):
            if adj_matrix[v][u]:  # edge exists
                if not visited[u]:
                    dfs(u, path)
                elif stack[u]:
                    cycle_start = path.index(u)
                    cycles.append(path[cycle_start:].copy())

        stack[v] = False
        path.pop()

    for node in range(n):
        if not visited[node]:
            dfs(node, [])

    return cycles



In [56]:
"""Give real world usage examples"""

uniform_debate_template = [
    "Uniforms promote equality",
    "Uniforms restrict personal expression",
    "Uniforms reduce bullying",
    "Uniforms do not promote equality",
    "Uniforms are not non beneficial"
]

uniform_initial_template = [
    "Uniforms are benefecial",
    "Uiforms are not benefecial"
]

animal_debate_template = [
    "killing animals is bad",
    "animals are tasty to eat"
]
animal_initial_template = [
    "killing living things is ok",
    "killing living things is not ok"
]

contradiction_check_template = ["i am awesome", "noone cares about me", "i suck"]

# Complex Circular Reasoning (Cycle)
circular_reasoning_nodes = [
    "This law is fair because it promotes justice.",
    "It promotes justice because the law ensures fairness.",
    "Ensuring fairness is the law's purpose because lawmakers intended it.",
    "Lawmakers intended the law to promote justice, which is fair."  # loops back to first node
]

# Complex Contradiction (Multi-branch)
contradiction_nodes = [
    "All industrial pollution should be banned to protect nature.",
    "Factory A’s pollution are desirable because it creates jobs.",  # contradicts first node
    "Factory B’s pollution are harmful and should be stopped.",       # consistent with first node
    "We should prioritize human welfare over environmental rules."    # introduces tension
]

# Combined Cycle + Contradiction
combined_cycle_contradiction_nodes = [
    "Hunting is unethical because killing is wrong.",
    "Killing for population control is ethical.",
    "Population control is necessary, so hunting is acceptable.",
    "Hunting is unethical because killing is wrong."  # cycle with contradiction
]

new_template = {
    "Speaker1": ["Hunting is unethical because killing is wrong.",
    "Killing for population control is ethical.",
    "Population control is necessary, so hunting is acceptable.",
    "Hunting is unethical because killing is wrong."],
    "Speaker2": []
}
EMPTY_TEMPLATE = []

In [57]:
print(list(new_template.values()))

[['Hunting is unethical because killing is wrong.', 'Killing for population control is ethical.', 'Population control is necessary, so hunting is acceptable.', 'Hunting is unethical because killing is wrong.'], []]


In [58]:
LABEL_DICT = {'ENTAILMENT': 1,'CONTRADICTION': -1, "NEUTRAL": 0}
NUM_SPEAKER = 1
DEBATE_TEMPLATE = circular_reasoning_nodes
INITIAL_TEMPLATE = EMPTY_TEMPLATE
STANCE_MODEL = pipeline("text-classification", model="roberta-large-mnli")



Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [ ]:
class Fallacy_Checker():
    
    def __init__(self, A_t, C_t):
        self.A_t = A_t
        self.C_t = C_t
        self.speaker_A_t = None
        
        assert self.C_t != []
        self.C_t =  np.array(self.C_t) #ARGS x NUM_SPEAKER
        speaker_arg_mask = np.transpose(self.C_t , (1,0)) #NUM_SPEAKER x ARGS
        self.A_t = np.array(self.A_t) #ARGS X ARGS
        self.speaker_A_t =  np.einsum('ij,jk->ijk', speaker_arg_mask, self.A_t) #NUM_SPEAKER x ARGS
        
    def contradiction(self):
        contradictions = {}
        for speaker in range(NUM_SPEAKER):
            for i in range(len(self.speaker_A_t[speaker])):
                for j in range(i, len(self.speaker_A_t[speaker])):
                    if((self.speaker_A_t[speaker][i][j], self.speaker_A_t[speaker][j][i]) in ((1,-1), (-1,-1), (-1,1))):
                        if(speaker in contradictions):
                            contradictions[speaker].append([i,j])
                        else:
                            contradictions[speaker] = [[i,j]]
                    
        if(len(contradictions)>0):
            return contradictions
        return None
    
    def circular_reasoning(self):
        cycles = {}
        
        for speaker in range(NUM_SPEAKER):
            x = self.speaker_A_t[speaker]
            x = np.where(x == 1, x, 0)
            cycles[speaker] = find_cycles_adj_matrix(x)
            
        if(len(cycles)>0):
            return cycles
        return None
        

class Debate():
    def __init__(self, initial_propositions, stance_model = STANCE_MODEL):
        self.claims = list(initial_propositions.values())
        self.num_speaker = len(initial_propositions.keys()) #No. of Speakers in the Debate
        self.n = 0 #Total number of claims in the Debate
        
        for user_claims in self.claims: #Sum over all user arguments
            self.n+=len(user_claims)
        
        #Adjacency matrix between every speaker claim initialized with 0's
        self.A_t = [[0 for _ in range(self.n)] for _ in range(self.n)]
        #Claim to Speaker Membership Matrix initialized with 0s
        self.C_t = [[0 for _ in range(self.num_speaker)] for _ in range(self.n)]
        
        for speaker_idx, speaker in enumerate(initial_propositions.keys()):
            for claim_idx, claim in  enumerate(initial_propositions[speaker]):
                self.C_t[claim_idx][speaker_idx] = 1
        

        self.nli_model = stance_model
        self.fallacy_checker = Fallacy_Checker(self.A_t, self.C_t)
        self.circular_reasoning = None
        self.contradictions = None

        
    def add_claim(self, speaker:int, claim:str):
        self.A_t.append([0 for _ in range(self.n)])
        self.n += 1
        self.claims[speaker].append(claim)
        self.C_t.append([0 for _ in range(self.num_speaker)])
        self.update_A_t()
        self.C_t[-1][speaker] = 1
    
    def update_A_t(self):
        claim_combinations = []
        all_claims 
        for speaker_claims in self.claims:
            for claim in self.claims:
                claim_combinations.append(f"{i} [SEP] {j}.")  
        
        pred = self.nli_model(claim_combinations)
        for idx, p in enumerate(pred):
            
            row = idx // self.n
            col = idx % self.n
            if col == self.n-1:
                self.A_t[row].append(LABEL_DICT[p["label"]])
            else:
                self.A_t[row][col] = LABEL_DICT[p["label"]]
    
    def check_fallacy(self):
        self.fallacy_checker = Fallacy_Checker(self.A_t, self.C_t)
        self.circular_reasoning = self.fallacy_checker.circular_reasoning()
        self.contradictions = self.fallacy_checker.contradiction()
        
        if(self.circular_reasoning):
            print("|----------The arguments have circular reasoning ----------|")
            print(self.circular_reasoning)
        
        if(self.contradictions):
            print("|----------The arguments have contradictions----------|")
            print(self.contradictions)
    
    def show_fallacy(self):
        if(self.circular_reasoning):
            for agent in self.circular_reasoning:
                print(f"|---------Circular Reasoning Fallacy of Agent: {agent}--------|")
                for fallacies in self.circular_reasoning[agent]:
                    s = ""
                    if(len(fallacies)>1):
                        s = " <-- "
                        for claim_idx in fallacies:
                            s += self.claims[claim_idx]
                            s += " --> "
                    if(s != ""):
                        print(s)
        
        if(self.contradictions):
            for agent in self.contradictions:
                print(f"|---------Contradiction Fallacy of Agent: {agent}--------------|")
                for fallacies in self.contradictions[agent]:
                    print(f" {self.claims[fallacies[0]]} != {self.claims[fallacies[1]]} ")
                    
                    


In [60]:
ROUNDS = len(DEBATE_TEMPLATE)
debate = Debate(initial_propositions = new_template)


In [61]:
print(debate.C_t)
print(debate.A_t)
print(debate.n)
debate.update_A_t()
print(debate.A_t)


[[1, 0], [1, 0], [1, 0], [1, 0]]
[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
4
[[1, 1, 1, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]


In [62]:
speaker = 0
for round in range(ROUNDS):
    # claim = str(input(f"Enter claim for speaker {speaker}: "))
    claim = contradiction_nodes[round]
    debate.add_claim(speaker, claim)
    speaker = (speaker+1)%NUM_SPEAKER

IndexError: list assignment index out of range

In [63]:
#print adjacency matrix that shows relationship between all claims
for i in debate.A_t:
    print(i)
print()
#Check claims and relationship between claims 
for i,j in zip(debate.A_t, debate.claims):
    print(j,i)

[1, 1, 0, 0, 1, 1, 1]
[0, 0, 1, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0, 0]

['Hunting is unethical because killing is wrong.', 'Killing for population control is ethical.', 'Population control is necessary, so hunting is acceptable.', 'Hunting is unethical because killing is wrong.'] [1, 1, 0, 0, 1, 1, 1]
[] [0, 0, 1, 0]
All industrial pollution should be banned to protect nature. [0, 0, 0, 0]
Factory A’s pollution are desirable because it creates jobs. [0, 0, 0, 0]


In [ ]:
debate.check_fallacy()

|----------The arguments have circular reasoning ----------|
{0: [[0], [0, 1], [1], [0, 3], [2], [3, 2], [3]]}


In [ ]:
debate.show_fallacy()

|---------Circular Reasoning Fallacy of Agent: 0--------|
 <-- This law is fair because it promotes justice. --> It promotes justice because the law ensures fairness. --> 
 <-- This law is fair because it promotes justice. --> Lawmakers intended the law to promote justice, which is fair. --> 
 <-- Lawmakers intended the law to promote justice, which is fair. --> Ensuring fairness is the law's purpose because lawmakers intended it. --> 
